In [ ]:
# feature_selection.ipynb
# Find the most relevant early-semester features for predicting student performance
import pandas as pd
import numpy as np
import os

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

In [ ]:
# Clone repo
!git clone https://github.com/KeXen1/Student-Performance-ML-Project.git

%cd Student-Performance-ML-Project

In [ ]:
# Load cleaned dataset
data = pd.read_csv("data/processed/cleaned_data.csv")

display(data.head())
print(data.shape)

In [ ]:
# Create performance category target from G3
def grade_category(g3):
    if g3 < 10:
        return "Low"
    elif g3 < 15:
        return "Medium"
    else:
        return "High"

data["performance_category"] = data["G3"].apply(grade_category)

print(data["performance_category"].value_counts())

In [ ]:
# Drop non-early-semester columns
X = data.drop(columns=["G3", "G2", "performance_category"], errors="ignore")
y = data["performance_category"]

print("Candidate features:")
print(X.columns.tolist())

In [ ]:
# Correlation analysis on numeric features against G3
numeric_data = data.select_dtypes(include=["int64", "float64"]).copy()
correlations = numeric_data.corr()["G3"].drop(["G3", "G2"]).sort_values(ascending=False)

print("Correlation of numeric features with G3:")
print(correlations)

In [ ]:
# Preprocessing for feature importance
categorical_features = X.select_dtypes(include=["object"]).columns
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

In [ ]:
# Random Forest feature importance
rf_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(n_estimators=100, random_state=42))
])

rf_pipeline.fit(X, y)

# Pull feature names
ohe = rf_pipeline.named_steps["preprocessor"].named_transformers_["cat"]
encoded_cat_features = ohe.get_feature_names_out(categorical_features)
all_feature_names = list(numeric_features) + list(encoded_cat_features)

importances = rf_pipeline.named_steps["classifier"].feature_importances_

feature_importance_df = pd.DataFrame({
    "Feature": all_feature_names,
    "Importance": importances
}).sort_values(by="Importance", ascending=False).reset_index(drop=True)

display(feature_importance_df.head(20))

In [ ]:
# Save feature importance ranking
os.makedirs("results", exist_ok=True)

feature_importance_df.to_csv("results/feature_importance.csv", index=False)

print("Feature importance saved to results/feature_importance.csv")

In [ ]:
# Selected early-semester features
# These are the features used for model training and prediction
selected_features = X.columns.tolist()

print("Selected early-semester features:")
for f in selected_features:
    print("-", f)

print()
print("Total features used:", len(selected_features))